# 01 — Data Cleaning & Anti-Leakage Imputation

**Project:** End-to-End Heart Disease Prediction System
**Stage:** 1 / 5 — Data Cleaning & Imputation

## Objective
Raw clinical data almost never arrives perfectly. In this dataset, `RestingBP` and
`Cholesterol` contain values of **0**, which are **physiologically impossible**
for a living patient. These are not "low" readings — they are missing-value
placeholders that were likely encoded as 0 during data collection.

This notebook performs three things, in order:

1. **Anomaly flagging** — before touching the data, we create a binary indicator
   (`*_was_zero`) so the model can learn *"this value was originally missing"*
   as a signal in itself, instead of silently losing that information.
2. **Anti-data-leakage imputation** — missing values are imputed using the
   **median computed strictly from the training set**. This median is then
   applied to both train and test. Computing statistics from the full
   (train + test) dataset is a common but subtle form of data leakage —
   it lets information about the test distribution influence training,
   inflating validation performance in a way that will not hold in production.
3. **Persisting clean, leak-free data** to `data/interim/` for the next stage.

## Output of this notebook
- `data/interim/train_imputed.csv`
- `data/interim/test_imputed.csv`

## 1. Imports & Configuration

In [1]:
import pandas as pd
import numpy as np

RAW_DIR = "../data/raw"
INTERIM_DIR = "../data/interim"

TRAIN_RAW_PATH = f"{RAW_DIR}/train.csv"
TEST_RAW_PATH  = f"{RAW_DIR}/test.csv"

## 2. Load Raw Data

In [2]:
df_train = pd.read_csv(TRAIN_RAW_PATH, sep=';')
df_test  = pd.read_csv(TEST_RAW_PATH,  sep=';')

print("Data loaded successfully!")
print(f"Train set: {df_train.shape[0]} rows, {df_train.shape[1]} columns")
print(f"Test set : {df_test.shape[0]} rows, {df_test.shape[1]} columns")
print("-" * 40)
df_train.head()

Data loaded successfully!
Train set: 734 rows, 13 columns
Test set : 184 rows, 12 columns
----------------------------------------


,id,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,1,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,2,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,3,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,4,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,5,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [3]:
df_train.describe().T

,count,mean,std,min,25%,50%,75%,max
id,734.0,461.389646,267.255825,1.0,227.5,461.5,693.75,918.0
Age,734.0,53.651226,9.364290,29.0,47.0,54.0,60.00,77.0
RestingBP,734.0,133.064033,18.438941,0.0,120.0,130.0,140.00,200.0
Cholesterol,734.0,199.683924,108.216855,0.0,177.0,223.0,269.00,603.0
FastingBS,734.0,0.227520,0.419517,0.0,0.0,0.0,0.00,1.0
MaxHR,734.0,136.178474,25.329254,60.0,118.0,138.0,155.00,202.0
Oldpeak,734.0,0.905041,1.082952,-2.6,0.0,0.6,1.50,6.2
HeartDisease,734.0,0.546322,0.498189,0.0,0.0,1.0,1.00,1.0


## 3. Combine for Consistent Processing

We tag each row with `is_train` and concatenate train + test. This lets us apply
**identical transformations** to both sets later (so column structure never
diverges), while still computing any *statistics* (like the median) from the
training partition only.

In [4]:
df_train['is_train'] = 1
df_test['is_train']  = 0
combined = pd.concat([df_train, df_test], axis=0, ignore_index=True)

print(f"Combined shape: {combined.shape}")

Combined shape: (918, 14)


## 4. Anomaly Flagging — Clinically Impossible Zero Values

`RestingBP == 0` (no blood pressure) and `Cholesterol == 0` are not valid
physiological readings for a living patient. We flag them **before** any
imputation so the information "this was originally missing" is preserved
as a feature.

In [5]:
n_bp_zero   = (combined['RestingBP'] == 0).sum()
n_chol_zero = (combined['Cholesterol'] == 0).sum()

print(f"RestingBP == 0   : {n_bp_zero} records")
print(f"Cholesterol == 0 : {n_chol_zero} records")

combined['RestingBP_was_zero']   = (combined['RestingBP']   == 0).astype(int)
combined['Cholesterol_was_zero'] = (combined['Cholesterol'] == 0).astype(int)

RestingBP == 0   : 1 records
Cholesterol == 0 : 172 records


## 5. Anti-Leakage Median Imputation

**Critical design decision:** the median used to fill missing values is computed
**only from rows where `is_train == 1`**, then applied to both train and test.
This mirrors how the model will behave in production, where future/unseen data
must be imputed using statistics learned during training — never from the data
being imputed itself.

In [6]:
median_bp = combined.loc[
    (combined['is_train'] == 1) & (combined['RestingBP'] > 0), 'RestingBP'
].median()

median_chol = combined.loc[
    (combined['is_train'] == 1) & (combined['Cholesterol'] > 0), 'Cholesterol'
].median()

combined.loc[combined['RestingBP']   == 0, 'RestingBP']   = median_bp
combined.loc[combined['Cholesterol'] == 0, 'Cholesterol'] = median_chol

print(f"RestingBP   imputed with train median   : {median_bp}")
print(f"Cholesterol imputed with train median   : {median_chol}")

RestingBP   imputed with train median   : 130.0
Cholesterol imputed with train median   : 237.0


## 6. Sanity Checks

In [7]:
assert (combined['RestingBP'] == 0).sum() == 0, "RestingBP still has zero values!"
assert (combined['Cholesterol'] == 0).sum() == 0, "Cholesterol still has zero values!"
assert combined['RestingBP_was_zero'].sum() == n_bp_zero, "Flag/imputation count mismatch (RestingBP)!"
assert combined['Cholesterol_was_zero'].sum() == n_chol_zero, "Flag/imputation count mismatch (Cholesterol)!"

print("[VALIDATION PASSED] No remaining zero anomalies. Flags consistent with original anomaly counts.")

[VALIDATION PASSED] No remaining zero anomalies. Flags consistent with original anomaly counts.


## 7. Split Back & Persist to `data/interim/`

In [8]:
train_imputed = combined[combined['is_train'] == 1].drop(columns=['is_train'])
test_imputed  = combined[combined['is_train'] == 0].drop(columns=['is_train'])

train_imputed.to_csv(f"{INTERIM_DIR}/train_imputed.csv", index=False)
test_imputed.to_csv(f"{INTERIM_DIR}/test_imputed.csv", index=False)

print("Stage 1 complete — clean, leak-free data persisted:")
print(f"  -> {INTERIM_DIR}/train_imputed.csv  ({train_imputed.shape[0]} rows, {train_imputed.shape[1]} cols)")
print(f"  -> {INTERIM_DIR}/test_imputed.csv   ({test_imputed.shape[0]} rows, {test_imputed.shape[1]} cols)")

Stage 1 complete — clean, leak-free data persisted:
  -> ../data/interim/train_imputed.csv  (734 rows, 15 cols)
  -> ../data/interim/test_imputed.csv   (184 rows, 15 cols)


---
**Next notebook:** `02_Exploratory_Data_Analysis.ipynb` — understanding distributions,
class balance, and correlations to inform the feature engineering decisions in
notebook 03.